# Data Preparation for Statistical Analysis

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve().parents[1]  # repository root (run from 02-papers/Result-01)
DATASETS_DIR = (Path("/home/ec2-user/studies") / "hpp_datasets").resolve()

sys.path.append(str(PROJECT_DIR))

popu_pfile = f'{DATASETS_DIR}/population/population.parquet'
anthro_pfile = f'{DATASETS_DIR}/anthropometrics/anthropometrics.parquet'

OUTPUT_PREFIX = 'outputs'
FIGURE_PREFIX = 'figures'

## Omics

In [ ]:
pfile ='../01-interim_data/omics_data/lipidomics_with_metadata.parquet'
df_lipid = pd.read_parquet(pfile)
print(df_lipid.shape)

metadata_col_lipid = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_lipid = [col for col in metadata_col_lipid if col in df_lipid.columns]
df_lipid_feature = df_lipid.drop(columns=metadata_col_lipid)
df_lipid_meta = df_lipid[metadata_col_lipid]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/gut_microbiome_with_metadata.parquet'
df_gut = pd.read_parquet(pfile)
print(df_gut.shape)

metadata_col_gut = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_gut = [col for col in metadata_col_gut if col in df_gut.columns]
df_gut_feature = df_gut.drop(columns=metadata_col_gut)
df_gut_meta = df_gut[metadata_col_gut]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/transcriptomics_with_metadata.parquet'
df_rna = pd.read_parquet(pfile)
print(df_rna.shape)

metadata_col_rna = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_rna = [col for col in metadata_col_rna if col in df_rna.columns]
df_rna_feature = df_rna.drop(columns=metadata_col_rna)
df_rna_meta = df_rna[metadata_col_rna]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/metabolomics_with_metadata.parquet'
df_metabo = pd.read_parquet(pfile)
print(df_metabo.shape)

metadata_col_metabo = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_metabo = [col for col in metadata_col_metabo if col in df_metabo.columns]
df_metabo_feature = df_metabo.drop(columns=metadata_col_metabo)
df_metabo_meta = df_metabo[metadata_col_metabo]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/oral_microbiome_with_metadata.parquet'
df_oral = pd.read_parquet(pfile)
print(df_oral.shape)

metadata_col_oral = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_oral = [col for col in metadata_col_oral if col in df_oral.columns]
df_oral_feature = df_oral.drop(columns=metadata_col_oral)
df_oral_meta = df_oral[metadata_col_oral]

# Dysregulated molecule

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import statsmodels.api as sm

# -----------------------------
# A. Helper: age stages
# -----------------------------
AGE_BINS = [
    (40, 45),      # baseline
    (45, 50),
    (50, 55),
    (55, 60),
    (60, 65),
    (65, 70),
]

omics_list = [[df_lipid_feature, df_lipid_meta], [df_gut_feature, df_gut_meta], [df_rna_feature, df_rna_meta], [df_metabo_feature, df_metabo_meta], [df_oral_feature, df_oral_meta]]
omics_type_list = ['lipid', 'gut_micro', 'rna', 'metabolomics', 'oral_micro']

def assign_age_stage(age: float) -> str:
    for lo, hi in AGE_BINS:
        # baseline 25–40 as in the reference paper (40 inclusive; later stages start at 40)
        if lo <= age < hi or (lo == 65 and age <= 70):
            return f"{lo}-{hi}"
    return np.nan

# -----------------------------
# B. Helper: confounder adjustment (residualization)
#    Linear regression per feature: feature ~ confounders
#    Returns the residuals
# -----------------------------
def residualize_features(df: pd.DataFrame, confounders: pd.DataFrame) -> pd.DataFrame:
    """
    df: rows=participants, cols=features
    confounders: rows=participants, cols=confounder vars (numeric or dummy variables)
    """
    # Add an intercept
    C = sm.add_constant(confounders, has_constant="add")
    resid = pd.DataFrame(index=df.index, columns=df.columns, dtype=float)

    # Per-feature regression (simple; large matrices could use a matrix solution)
    for feat in df.columns:
        y = df[feat].astype(float)
        # Drop rows with missing values
        mask = y.notna() & C.notna().all(axis=1)
        if mask.sum() < 10:
            resid[feat] = np.nan
            continue
        model = sm.OLS(y[mask].values, C.loc[mask].values).fit()
        # residual = y - y_hat
        y_hat = model.predict(C.loc[mask].values)
        r = pd.Series(index=df.index, dtype=float)
        r.loc[mask] = y[mask].values - y_hat
        resid[feat] = r

    return resid

# -----------------------------
# C. Helper: Wilcoxon test
#    Two independent groups: Mann–Whitney U (equivalent to the Wilcoxon rank-sum test)
# -----------------------------
def wilcoxon_rank_sum(group_a: np.ndarray, group_b: np.ndarray) -> float:
    a = group_a[~np.isnan(group_a)]
    b = group_b[~np.isnan(group_b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan
    # two-sided
    stat = mannwhitneyu(a, b, alternative="two-sided")
    return stat.pvalue

# -----------------------------
# D. Helper: permutation p-value
#    Shuffle labels and recompute abs(mean difference)
# -----------------------------
def permutation_pvalue(a: np.ndarray, b: np.ndarray, n_perm: int = 1000, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan

    observed = abs(a.mean() - b.mean())
    pooled = np.concatenate([a, b])
    n_a = len(a)

    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        a_perm = pooled[:n_a]
        b_perm = pooled[n_a:]
        diff = abs(a_perm.mean() - b_perm.mean())
        if diff >= observed:
            count += 1

    # Add-one smoothing to avoid zero
    return (count + 1) / (n_perm + 1)

# -----------------------------
# E. Main workflow: statistics table for Fig. 1e
# -----------------------------
def run_stagewise_dysregulation(
    X: pd.DataFrame,          # rows=sample_id, cols=features
    meta: pd.DataFrame,       # must include sample_id index OR column; participant_id, age, healthy
    confounder_cols: list,    # e.g. ["BMI", "sex", "ethnicity", "IRIS"]
    n_perm: int = 1000,
    seed: int = 0
) -> pd.DataFrame:
    # 1) Align indices
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    meta = meta.loc[X.index].copy()

    # 2) Keep healthy visits only
    # keep = meta["healthy"] == True
    # Xh = X.loc[keep].copy()
    # mh = meta.loc[keep].copy()

    Xh = X.copy()
    mh = meta.copy()

    # 3) Aggregate to participant level (mean over healthy samples of the same person)
    #    with a representative age per participant (mean age)
    Xp = Xh.join(mh[["participant_id"]]).groupby("participant_id").mean(numeric_only=True)
    age_p = mh.groupby("participant_id")["age"].mean()
    stages = age_p.apply(assign_age_stage)

    # 4) Participant-level confounders
    #    One-hot encode categorical variables (sex, ethnicity, ...)
    conf = mh.groupby("participant_id")[confounder_cols].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
    conf = pd.get_dummies(conf, drop_first=True, dtype=float)

    # 5) Confounder adjustment: feature ~ confounders, keep residuals
    Xr = residualize_features(Xp, conf)

    # baseline participants
    baseline_stage = "40-45"
    baseline_ids = stages.index[stages == baseline_stage]
    if len(baseline_ids) < 5:
        raise ValueError("Baseline(40–45) participant too few after filtering.")

    # 6) Test each feature in each age stage against baseline
    stage_list = [f"{lo}-{hi}" for (lo, hi) in AGE_BINS if not (lo == 25 and hi == 40)]
    # stage_list should contain 6 comparisons (Bonferroni × 6)
    results = []

    for feat in Xr.columns:
        base_vals = Xr.loc[baseline_ids, feat].values.astype(float)

        for st in stage_list:
            stage_ids = stages.index[stages == st]
            stage_vals = Xr.loc[stage_ids, feat].values.astype(float)

            p_wil = wilcoxon_rank_sum(stage_vals, base_vals)
            p_perm = permutation_pvalue(stage_vals, base_vals, n_perm=n_perm, seed=seed)

            results.append({
                "feature": feat,
                "stage": st,
                "n_stage": int(np.isfinite(stage_vals).sum()),
                "n_baseline": int(np.isfinite(base_vals).sum()),
                "mean_diff_abs": float(abs(np.nanmean(stage_vals) - np.nanmean(base_vals))),
                "p_wilcoxon": p_wil,
                "p_perm": p_perm,
            })

    res = pd.DataFrame(results)

    # 7) Bonferroni across the 6 stage comparisons of each feature: p * 6
    res["p_wilcoxon_bonf6"] = (res["p_wilcoxon"] * 6).clip(upper=1.0)
    res["p_perm_bonf6"] = (res["p_perm"] * 6).clip(upper=1.0)

    # 8) Flag dysregulated features (P < 0.05)
    res["dysregulated_wilcoxon"] = res["p_wilcoxon_bonf6"] < 0.05
    res["dysregulated_perm"] = res["p_perm_bonf6"] < 0.05

    return res

# -----------------------------
# F. (Optional) Heat-map matrix for Fig. 1g: feature × stage z-score / mean shift
# -----------------------------
def make_heatmap_matrix(res: pd.DataFrame, value_col="mean_diff_abs") -> pd.DataFrame:
    """
    The signed difference in mean(residual) between each stage and baseline is an alternative.
    abs diff is used here for illustration; a signed difference is recommended.
    """
    mat = res.pivot(index="feature", columns="stage", values=value_col)
    # Standardize to z-scores (per feature row)
    mat_z = mat.sub(mat.mean(axis=1), axis=0).div(mat.std(axis=1).replace(0, np.nan), axis=0)
    return mat_z

res_dys_list = [run_stagewise_dysregulation(omics[0], omics[1], confounder_cols=['sex', 'bmi']) for omics in omics_list]

res_dys_typed = [
    df.assign(type=omics_type_list[i])
    for i, df in enumerate(res_dys_list)
]

df_res_dys = pd.concat(res_dys_typed, axis=0, ignore_index=True)
df_res_dys.to_parquet(f'{OUTPUT_PREFIX}/dysregulated_molecules.parquet')
df_res_dys.to_csv(f'{OUTPUT_PREFIX}/dysregulated_molecules.csv', index=False)

In [ ]:
df_res_dys.to_parquet(f'{OUTPUT_PREFIX}/dysregulated_molecules.parquet')
pd.read_parquet(f'{OUTPUT_PREFIX}/dysregulated_molecules.parquet')

In [ ]:
# Unique list of dysregulated features
dysregulated_features = (
    df_res_dys.loc[df_res_dys["dysregulated_wilcoxon"], "feature"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

# Save as CSV
dysregulated_features.to_frame(name="feature").to_csv(
    f"{OUTPUT_PREFIX}/dysregulated_wilcoxon_features.csv",
    index=False
)

# Hierarchical clustering

In [ ]:
dysregulated_features = pd.read_csv(
    f"{OUTPUT_PREFIX}/dysregulated_wilcoxon_features.csv"
)["feature"].tolist()

META_COLS = ['participant_id', 'cohort', 'research_stage', 'array_index',
             'collection_date', 'collection_timestamp', 'timezone',
             'year_of_birth', 'month_of_birth', 'collection_datetime',
             'batch', 'data_source', 'hmo', 'bmi', 'age', 'sex']

rna_dys = [feat for feat in df_rna.columns if feat in dysregulated_features + META_COLS]
df_rna_dys = df_rna[rna_dys].copy()
metabo_dys = [feat for feat in df_metabo.columns if feat in dysregulated_features + META_COLS]
df_metabo_dys = df_metabo[metabo_dys].copy()
lipid_dys = [feat for feat in df_lipid.columns if feat in dysregulated_features + META_COLS]
df_lipid_dys = df_lipid[lipid_dys].copy()
gut_dys = [feat for feat in df_gut.columns if feat in dysregulated_features + META_COLS]
df_gut_dys = df_gut[gut_dys].copy()
oral_dys = [feat for feat in df_oral.columns if feat in dysregulated_features + META_COLS]
df_oral_dys = df_oral[oral_dys].copy()

df_rna_dys.to_parquet(f'../01-interim_data/omics_data/transcriptomics_dysregulated.parquet', index=False)
df_rna_dys.to_csv(f'../01-interim_data/omics_data/transcriptomics_dysregulated.csv', index=False)
df_metabo_dys.to_parquet(f'../01-interim_data/omics_data/metabolomics_dysregulated.parquet', index=False)
df_metabo_dys.to_csv(f'../01-interim_data/omics_data/metabolomics_dysregulated.csv', index=False)
df_lipid_dys.to_parquet(f'../01-interim_data/omics_data/lipidomics_dysregulated.parquet', index=False)
df_lipid_dys.to_csv(f'../01-interim_data/omics_data/lipidomics_dysregulated.csv', index=False)
df_gut_dys.to_parquet(f'../01-interim_data/omics_data/gut_microbiome_dysregulated.parquet', index=False)
df_gut_dys.to_csv(f'../01-interim_data/omics_data/gut_microbiome_dysregulated.csv', index=False)
df_oral_dys.to_parquet(f'../01-interim_data/omics_data/oral_microbiome_dysregulated.parquet', index=False)
df_oral_dys.to_csv(f'../01-interim_data/omics_data/oral_microbiome_dysregulated.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LinearRegression
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════
# 1. Configuration
# ══════════════════════════════════════════════════════════════
META_COLS = ['participant_id', 'cohort', 'research_stage', 'array_index',
             'collection_date', 'collection_timestamp', 'timezone',
             'year_of_birth', 'month_of_birth', 'collection_datetime',
             'batch', 'data_source', 'hmo', 'bmi', 'age', 'sex']

CONFOUNDERS  = ['bmi', 'sex']   # confounders to regress out
AGE_MIN, AGE_MAX = 40, 70
N_AGE_BINS   = 30               # number of age bins (columns)
MIN_SAMPLES  = 20               # minimum samples per age bin

OMICS_COLORS = {
    'Transcriptomics': '#BF4E50',
    'Metabolomics':    '#4797D6',
    'Lipidomics':      '#2F9B7C',
    'Gut Microbiome':  '#D68B40',
    'Oral Microbiome': '#6F4E9C',
}

MICROBIOME = {'Gut Microbiome', 'Oral Microbiome'}

df_rna_dys = pd.read_csv(f'../01-interim_data/omics_data/transcriptomics_dysregulated.csv')
df_metabo_dys = pd.read_csv(f'../01-interim_data/omics_data/metabolomics_dysregulated.csv')
df_lipid_dys = pd.read_csv(f'../01-interim_data/omics_data/lipidomics_dysregulated.csv')
df_gut_dys = pd.read_csv(f'../01-interim_data/omics_data/gut_microbiome_dysregulated.csv')
df_oral_dys = pd.read_csv(f'../01-interim_data/omics_data/oral_microbiome_dysregulated.csv')

# Omics data
OMICS = {
    'Transcriptomics': df_rna_dys,
    'Metabolomics':    df_metabo_dys,
    'Lipidomics':      df_lipid_dys,
    'Gut Microbiome':  df_gut_dys,
    'Oral Microbiome': df_oral_dys,
}

# ══════════════════════════════════════════════════════════════
# 2. CLR transform (microbiome)
# ══════════════════════════════════════════════════════════════
def clr_transform(X: np.ndarray) -> np.ndarray:
    X = X.astype(float) + 1.0
    X = X / X.sum(axis=1, keepdims=True)
    X = np.log(X)
    return X - X.mean(axis=1, keepdims=True)

# ══════════════════════════════════════════════════════════════
# 3. Regress confounders out of each feature and keep the residuals
# ══════════════════════════════════════════════════════════════
def regress_out_confounders(df: pd.DataFrame,
                            feat_cols: list,
                            confounders: list) -> pd.DataFrame:
    """
    Fit an independent linear regression for each feature column:
        feature ~ bmi + sex
    and return the residuals as adjusted values.
    """
    avail_conf = [c for c in confounders if c in df.columns]
    if not avail_conf:
        print("  Warning: Confounder columns not found, skipping regression correction")
        return df[feat_cols].copy()

    cov_mat = df[avail_conf].values.astype(float)
    resid_df = pd.DataFrame(index=df.index, columns=feat_cols, dtype=float)

    for feat in feat_cols:
        y    = df[feat].values.astype(float)
        mask = ~(np.isnan(y) | np.any(np.isnan(cov_mat), axis=1))
        if mask.sum() < MIN_SAMPLES:
            resid_df[feat] = np.nan
            continue
        reg = LinearRegression().fit(cov_mat[mask], y[mask])
        resid         = np.full(len(y), np.nan)
        resid[mask]   = y[mask] - reg.predict(cov_mat[mask])
        resid_df[feat] = resid

    return resid_df

# ══════════════════════════════════════════════════════════════
# 4. Bin by age and average within bins → rows = features, columns = age bins
# ══════════════════════════════════════════════════════════════
age_bins   = np.linspace(AGE_MIN, AGE_MAX, N_AGE_BINS + 1)
age_labels = [f'{int(age_bins[i])}–{int(age_bins[i+1])}'
              for i in range(N_AGE_BINS)]

def compute_age_profile(df: pd.DataFrame,
                        resid_df: pd.DataFrame,
                        feat_cols: list) -> pd.DataFrame:
    """
    Bin participants by age and compute the mean of each feature within each bin.
    Returns a DataFrame: rows = features, columns = age bins.
    """
    age   = df['age'].values
    bins  = np.digitize(age, age_bins) - 1
    bins  = np.clip(bins, 0, N_AGE_BINS - 1)

    profile = np.full((len(feat_cols), N_AGE_BINS), np.nan)
    for b in range(N_AGE_BINS):
        mask = bins == b
        if mask.sum() < MIN_SAMPLES:
            continue
        seg = resid_df.iloc[mask][feat_cols].values.astype(float)
        profile[:, b] = np.nanmean(seg, axis=0)

    return pd.DataFrame(profile,
                        index=feat_cols,
                        columns=age_labels)

# ══════════════════════════════════════════════════════════════
# 5. Z-score standardization (per row, i.e. per feature)
# ══════════════════════════════════════════════════════════════
def zscore_rows(mat: np.ndarray) -> np.ndarray:
    """
    Z-score each row of the matrix:
        z = (x - row_mean) / row_std
    so that all features share one scale and colour shows deviation from each feature's own mean.
    """
    mu  = np.nanmean(mat, axis=1, keepdims=True)
    std = np.nanstd(mat,  axis=1, keepdims=True)
    std[std < 1e-10] = 1.0   # avoid division by zero
    return (mat - mu) / std

# ══════════════════════════════════════════════════════════════
# 6. Main workflow: process all omics layers
# ══════════════════════════════════════════════════════════════
print("Step 1: Regression correction + age profile computation...")

all_profiles = []   # one row per feature: standardized age profile
all_omics    = []   # omics layer of each feature
all_features = []   # feature names

for oname, df_raw in OMICS.items():
    print(f"  [{oname}]", end=' ')
    df = df_raw.copy()
    df['participant_id'] = df['participant_id'].astype(str)

    # Filter by age range
    df = df[(df['age'] >= AGE_MIN) & (df['age'] <= AGE_MAX)].copy()
    df = df.dropna(subset=['age'])

    feat_cols = [c for c in df.columns if c not in META_COLS]

    # CLR transform for microbiome layers
    X = df[feat_cols].values.astype(float)
    if oname in MICROBIOME:
        X = clr_transform(X)
        df[feat_cols] = X

    # Drop low-variance features
    col_std   = np.nanstd(X, axis=0)
    valid_idx = col_std > 1e-10
    feat_cols = [f for f, v in zip(feat_cols, valid_idx) if v]
    df        = df[['participant_id', 'age'] + CONFOUNDERS + feat_cols
                   ].copy()
    df        = df.dropna(subset=CONFOUNDERS)

    # Confounder adjustment
    resid_df  = regress_out_confounders(df, feat_cols, CONFOUNDERS)

    # Age profile
    profile   = compute_age_profile(df, resid_df, feat_cols)

    # Drop all-NaN rows
    valid_rows = profile.notna().any(axis=1)
    profile    = profile[valid_rows]
    feat_cols_valid = profile.index.tolist()

    all_profiles.append(profile)
    all_omics.extend([oname] * len(feat_cols_valid))
    all_features.extend(feat_cols_valid)
    print(f"{len(feat_cols_valid)} features")

# Combine the profile matrices of all omics layers
profile_mat = pd.concat(all_profiles, axis=0)
profile_mat.index = all_features

print(f"\nTotal features: {len(profile_mat)}")

# ══════════════════════════════════════════════════════════════
# 7. Z-score standardization
# ══════════════════════════════════════════════════════════════
print("Step 2: Z-score standardization...")

mat_raw   = profile_mat.values.astype(float)

# Fill missing values with row means (clustering requires complete data)
row_means = np.nanmean(mat_raw, axis=1, keepdims=True)
mat_filled = np.where(np.isnan(mat_raw), row_means, mat_raw)

mat_z     = zscore_rows(mat_filled)

# ══════════════════════════════════════════════════════════════
# 8. Hierarchical clustering (rows)
# ══════════════════════════════════════════════════════════════
print("Step 3: hierarchical clustering (row clustering)...")

# Correlation distance groups features with similar trends
dist_mat  = pdist(mat_z, metric='correlation')
dist_mat  = np.nan_to_num(dist_mat, nan=2.0)   # NaN distances set to the maximum, 2
Z_linkage = linkage(dist_mat, method='ward')
row_order = leaves_list(Z_linkage)

mat_plot      = mat_z[row_order, :]
omics_ordered = [all_omics[i] for i in row_order]
feat_ordered  = [all_features[i] for i in row_order]

# Save the matrices for later use
# np.save(f'{OUTPUT_PREFIX}/age_trend_matrix.npy', mat_plot)
pd.DataFrame(mat_plot,
             index   = feat_ordered,
             columns = age_labels).to_csv(
    f'{OUTPUT_PREFIX}/age_trend_matrix_dys.csv')

pd.DataFrame({'feature': feat_ordered,
              'omics':   omics_ordered}).to_csv(
    f'{OUTPUT_PREFIX}/age_trend_row_info_dys.csv', index=False)

print(f"\nDone! Output: ")
print(f"  {FIGURE_PREFIX}/age_trend_heatmap_dysregulated.png")
print(f"  {FIGURE_PREFIX}/age_trend_heatmap_dysregulated.pdf")
print(f"  {OUTPUT_PREFIX}/age_trend_matrix_dys.csv     Standardized matrix")
print(f"  {OUTPUT_PREFIX}/age_trend_row_info_dys.csv   Row information (feature names + omics)")